# Domain 3: Claude Code Configuration & Workflows
## Task Statement 3.1: Configure CLAUDE.md Files with Appropriate Hierarchy and Scoping
**Certification Weight**: 18% Exam Weight | **Standard**: Claude 4.5 & Extended Thinking

## 🎯 Exam Objective
Understand the hierarchical resolution of `CLAUDE.md` configuration files, file size budgeting (<200 lines), `@import` modular composition, and precedence rules across user, project, and directory levels.

---

## 🏗️ Architecture: Configuration Hierarchy & Resolution Precedence

```mermaid
flowchart TD
    UserGlobal["1. User Global Scope<br>~/.claude/CLAUDE.md<br>(Developer defaults, personal shell aliases)"]
    
    ProjRoot["2. Project Root Scope<br>./CLAUDE.md or .claude/CLAUDE.md<br>(Team build commands, architecture rules, <200 lines)"]
    
    DirRules["3. Path-Scoped Rules<br>.claude/rules/*.md<br>(Activated conditionally by glob matching)"]
    
    SubdirScope["4. Subdirectory Scope<br>packages/frontend/CLAUDE.md<br>(Sub-package specific test/lint configs)"]
    
    UserGlobal -->|"Overridden by"| ProjRoot
    ProjRoot -->|"Overridden by"| DirRules
    DirRules -->|"Overridden by"| SubdirScope
    
    SubdirScope --> FinalMerged["Final Merged Active Context<br>(Highest specificity wins)"]
```

---

## ⚠️ Critical Exam Distinctions & Anti-Patterns

| Architecture Concept | Best Practice Implementation | Exam Trap to Reject |
|---|---|---|
| **Root File Sizing** | Keep `CLAUDE.md` **under 200 lines**. Offload specialized rules to `.claude/rules/` and workflows to `.claude/skills/`. | **TRAP**: Creating a monolithic 1,500-line `CLAUDE.md` that eats substantial context window tokens on every turn. |
| **Precedence Order** | **Local overrides global**: Subdirectory / path rules take precedence over project root, which takes precedence over user config. | **TRAP**: Assuming user-global configs silently override project-specific team conventions. |
| **Modular Rule Inclusion** | Use `@import` syntax or `.claude/rules/*.md` with YAML frontmatter `paths` globs to load on-demand. | **TRAP**: Duplicating coding standards across 10 different files or hardcoding every rule globally. |

---

## 💡 Architectural Decision Matrix

| Configuration Scope | Path | Best Suited For | Loaded When |
|---|---|---|---|
| **User Global** | `~/.claude/CLAUDE.md` | Personal developer preferences, editor configs | Always loaded across all projects. |
| **Project Root** | `./CLAUDE.md` | Build commands, testing framework, high-level guidelines (<200 lines) | Always loaded in current project. |
| **Path-Scoped Rule** | `.claude/rules/api.md` | API conventions, database patterns | Loaded only when files matching `paths` are edited. |
| **Subdirectory** | `packages/ui/CLAUDE.md` | Framework-specific UI component guidelines | Loaded when working within that directory. |

---

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards (October 2026)
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns. Stripping thinking blocks causes an API `400 Bad Request` error.
- **Case Facts Preservation**: Immutable transactional data (Order IDs, customer accounts, amounts) must be maintained in a dedicated Case Facts block outside conversation summaries.

In [ ]:
# Step 1: Environment & Setup
import fnmatch
import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 3.1 Claude Code Workflows.")

### 📜 Step 2: Path-Scoped Rules Engine & CI Review Simulator
We define the rule store and implement rule matching and CI review execution.

In [ ]:
# Step 2: Rule Matching & CI Simulator
@dataclass
class PathScopedRule:
    rule_name: str
    globs: List[str]
    rule_content: str

    def matches(self, filepath: str) -> bool:
        return any(fnmatch.fnmatch(filepath, pattern) for pattern in self.globs)

RULE_STORE: List[PathScopedRule] = [
    PathScopedRule(
        rule_name="api_conventions",
        globs=["src/api/**/*", "**/api/**/*.py"],
        rule_content="All API handlers must use explicit Pydantic response models and async/await.",
    ),
    PathScopedRule(
        rule_name="test_conventions",
        globs=["**/*.test.*", "tests/**/*.py"],
        rule_content="All tests must use pytest patterns and include docstrings with AAA structure.",
    ),
]

def resolve_active_rules_for_file(filepath: str) -> List[str]:
    """Determines which rules apply to a given file path."""
    active = []
    for rule in RULE_STORE:
        if rule.matches(filepath):
            active.append(rule.rule_content)
    return active

def simulate_ci_review_execution(git_diff: str, active_rules: List[str], is_non_interactive: bool = True) -> Dict[str, Any]:
    """Simulates headless CI audit (-p mode). Fails if interactive input is expected."""
    if not is_non_interactive:
        raise RuntimeError("Missing -p flag: CI/CD runners must execute in non-interactive mode!")
    
    findings = []
    if "eval(" in git_diff or "exec(" in git_diff:
        findings.append({
            "severity": "CRITICAL",
            "rule": "SEC001_NO_DYNAMIC_EXECUTION",
            "message": "Dynamic code execution (eval/exec) strictly prohibited in pull requests."
        })
    return {
        "status": "completed",
        "total_findings": len(findings),
        "findings": findings,
        "exit_code": 1 if findings else 0,
    }

print(" Rule matcher and CI reviewer initialized.")

### 🧪 Step 3: Interactive Simulation
1. Test rule resolution for test vs API files.
2. Simulate CI review of a PR containing dangerous `eval()` code.

In [ ]:
# Step 3: Simulation
# 1. Rule Resolution
test_rules = resolve_active_rules_for_file("tests/unit/test_auth.py")
print("Active Rules for test_auth.py:", test_rules)

api_rules = resolve_active_rules_for_file("src/api/v1/orders.py")
print("Active Rules for orders.py:", api_rules)

# 2. CI/CD Audit Simulation
unsafe_diff = """
@@ -10,3 +10,4 @@
 def handle_request(raw_code):
+    return eval(raw_code)
"""
ci_result = simulate_ci_review_execution(unsafe_diff, active_rules=api_rules, is_non_interactive=True)
print("\nCI Review Result:", json.dumps(ci_result, indent=2))

### 🔬 Step 4: Verification Assertions
We assert that:
1. Test rules apply only to test paths.
2. API rules apply only to API paths.
3. CI review correctly flags `eval()` as critical with exit code 1.
4. Non-interactive mode enforcement raises RuntimeError if flag is missing.

In [ ]:
# Step 4: Verification Checks
assert any("pytest" in r for r in test_rules)
assert any("Pydantic" in r for r in api_rules)

assert ci_result["exit_code"] == 1
assert ci_result["findings"][0]["severity"] == "CRITICAL"

# Verify non-interactive safeguard
try:
    simulate_ci_review_execution(unsafe_diff, api_rules, is_non_interactive=False)
    assert False, "Should have failed without non-interactive mode!"
except RuntimeError as exc:
    assert "Missing -p flag" in str(exc)

print(" All verification assertions passed! Configuration hierarchy verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Know the 5 hierarchy levels: Enterprise > User Global > Project Root > Path Rules > CLI Flags.
- **Rule 2**: Use path-specific rules to conditionally load conventions only when matching files are edited.
- **Rule 3**: In CI/CD pipelines, always pass `-p` / `--non-interactive` to prevent hanging processes.